# **Pharmacy Transactions Data Cleaning**

## 1. Importing Libraries and Loading Data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import sys
sys.path.append(str(Path.cwd().parent))

from scripts.paths import RAW_DIR, CLEANED_DIR

df = pd.read_csv(RAW_DIR / "08_pharmacy_transactions_dirty.csv")

## 2. Initial Data Inspection

In [3]:
df.head()

,Pharmacy_Txn_ID,Patient_ID,Medicine_ID,Encounter_ID,Txn_Date,Quantity,Unit_Price,Discount,Total_Amount
0,PT500000,P107035,M1091,E205571,2025-10-11,13,1153.50,212.49,14783.01
1,PT500001,P105797,M1119,E208474,2026-04-01,-1,898.72,475.10,-1373.82
2,PT500002,P101642,M1088,E213936,2026-08-16,27,2874.78,94.26,77524.80
3,PT500003,P101071,M1094,E212748,2025-08-29,24,1422.18,303.45,33828.87
4,PT500004,P103131,M1054,E201917,2025-07-07,3,1475.38,122.44,4303.70


In [4]:
df.shape

(14000, 9)

In [5]:
df.columns.tolist()

['Pharmacy_Txn_ID',
 'Patient_ID',
 'Medicine_ID',
 'Encounter_ID',
 'Txn_Date',
 'Quantity',
 'Unit_Price',
 'Discount',
 'Total_Amount']

In [6]:
df.dtypes

Pharmacy_Txn_ID     object
Patient_ID          object
Medicine_ID         object
Encounter_ID        object
Txn_Date            object
Quantity             int64
Unit_Price         float64
Discount           float64
Total_Amount       float64
dtype: object

In [7]:
df.isna().sum()

Pharmacy_Txn_ID     0
Patient_ID          0
Medicine_ID        99
Encounter_ID        0
Txn_Date            0
Quantity            0
Unit_Price          0
Discount            0
Total_Amount        0
dtype: int64

In [8]:
df.duplicated().sum()

0

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14000 entries, 0 to 13999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Pharmacy_Txn_ID  14000 non-null  object 
 1   Patient_ID       14000 non-null  object 
 2   Medicine_ID      13901 non-null  object 
 3   Encounter_ID     14000 non-null  object 
 4   Txn_Date         14000 non-null  object 
 5   Quantity         14000 non-null  int64  
 6   Unit_Price       14000 non-null  float64
 7   Discount         14000 non-null  float64
 8   Total_Amount     14000 non-null  float64
dtypes: float64(3), int64(1), object(5)
memory usage: 984.5+ KB


In [12]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Pharmacy_Txn_ID,14000,14000,PT500000,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Patient_ID,14000,6642,P106405,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Medicine_ID,13901,182,M1135,107,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Encounter_ID,14000,9123,E203964,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Txn_Date,14000,601,2025-11-08,39,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,14000.0,NaN,NaN,NaN,13.464071,9.81579,-3.0,5.0,13.0,22.0,30.0
Unit_Price,14000.0,NaN,NaN,NaN,1502.578153,864.015934,5.19,747.24,1505.79,2251.2675,2999.8
Discount,14000.0,NaN,NaN,NaN,251.293199,144.125381,0.0,127.435,254.185,374.5775,499.98
Total_Amount,14000.0,NaN,NaN,NaN,19961.754439,20686.667148,-9403.25,3303.62,13698.22,31794.99,89619.81


## 3. Audit of Specific Columns

### 3.1 Identifier Columns

In [13]:
for col in ["Pharmacy_Txn_ID", "Patient_ID", "Medicine_ID", "Encounter_ID"]:
    s = df[col].astype("string")
    print(
        col,
        "| missing:", s.isna().sum(),
        "| duplicates:", s.duplicated().sum(),
        "| leading/trailing whitespace:", (s.notna() & (s != s.str.strip())).sum()
    )

print("Pharmacy_Txn_ID invalid pattern:",
      (~df["Pharmacy_Txn_ID"].astype("string").fillna("").str.fullmatch(r"PT\d{6}")).sum())

print("Patient_ID invalid pattern:",
      (~df["Patient_ID"].astype("string").fillna("").str.fullmatch(r"P\d{6}")).sum())

print("Medicine_ID invalid pattern:",
      (~df["Medicine_ID"].astype("string").fillna("").str.fullmatch(r"M\d{4}")).sum())

print("Encounter_ID invalid pattern:",
      (~df["Encounter_ID"].astype("string").fillna("").str.fullmatch(r"E\d{6}")).sum())


Pharmacy_Txn_ID | missing: 0 | duplicates: 0 | leading/trailing whitespace: 0
Patient_ID | missing: 0 | duplicates: 7358 | leading/trailing whitespace: 0
Medicine_ID | missing: 99 | duplicates: 13817 | leading/trailing whitespace: 0
Encounter_ID | missing: 0 | duplicates: 4877 | leading/trailing whitespace: 0
Pharmacy_Txn_ID invalid pattern: 0
Patient_ID invalid pattern: 0
Medicine_ID invalid pattern: 139
Encounter_ID invalid pattern: 0


### 3.2 Medicine Reference Integrity

In [14]:
medicine_reference_path = CLEANED_DIR / "07_medicines_cleaned.csv"

if medicine_reference_path.exists():
    medicines_df = pd.read_csv(medicine_reference_path)
    valid_medicine_ids = set(medicines_df["Medicine_ID"].dropna().astype(str))

    invalid_medicine_mask = (
        df["Medicine_ID"].notna()
        & ~df["Medicine_ID"].astype(str).isin(valid_medicine_ids)
    )

    print("Medicine IDs not found in medicine master:", invalid_medicine_mask.sum())
    print(df.loc[invalid_medicine_mask, "Medicine_ID"].value_counts(dropna=False))
else:
    medicines_df = None
    valid_medicine_ids = set()
    print("Medicine master file not found:", medicine_reference_path)
    print("Pattern-based Medicine_ID audit is used instead.")


Medicine master file not found: E:\hospital_management_project\data\cleaned\07_medicines_cleaned.csv
Pattern-based Medicine_ID audit is used instead.


### 3.3 Encounter and Patient Reference Integrity

In [15]:
encounter_reference_path = CLEANED_DIR / "05_encounters_cleaned.csv"

if encounter_reference_path.exists():
    encounters_df = pd.read_csv(encounter_reference_path)

    encounter_patient_map = (
        encounters_df
        .drop_duplicates("Encounter_ID")
        .set_index("Encounter_ID")["Patient_ID"]
    )

    linked_patient = df["Encounter_ID"].map(encounter_patient_map)

    print("Encounter IDs missing from encounter master:",
          linked_patient.isna().sum())

    patient_mismatch_mask = (
        linked_patient.notna()
        & df["Patient_ID"].notna()
        & (df["Patient_ID"].astype("string") != linked_patient.astype("string"))
    )

    print("Patient_ID / Encounter_ID mismatches:",
          patient_mismatch_mask.sum())
else:
    encounters_df = None
    encounter_patient_map = pd.Series(dtype="string")
    linked_patient = pd.Series(index=df.index, dtype="string")
    patient_mismatch_mask = pd.Series(False, index=df.index)
    print("Encounter master file not found:", encounter_reference_path)


Encounter IDs missing from encounter master: 0
Patient_ID / Encounter_ID mismatches: 13999


### 3.4 Transaction Date

In [17]:
txn_date_audit = pd.to_datetime(df["Txn_Date"], errors="coerce")

print("Invalid/unparseable transaction dates:", txn_date_audit.isna().sum())
print("Minimum transaction date:", txn_date_audit.min())
print("Maximum transaction date:", txn_date_audit.max())

Invalid/unparseable transaction dates: 0
Minimum transaction date: 2025-01-01 00:00:00
Maximum transaction date: 2026-08-24 00:00:00


### 3.5 Quantity

In [19]:
print("Minimum Quantity:", df["Quantity"].min())
print("Maximum Quantity:", df["Quantity"].max())
print("Zero Quantity:", (df["Quantity"] == 0).sum())
print("Negative Quantity:", (df["Quantity"] < 0).sum())
print("Non-positive Quantity:", (df["Quantity"] <= 0).sum())

Minimum Quantity: -3
Maximum Quantity: 30
Zero Quantity: 420
Negative Quantity: 1215
Non-positive Quantity: 1635


### 3.6 Unit Price and Discount

In [20]:
print("Unit_Price missing:", df["Unit_Price"].isna().sum())
print("Unit_Price <= 0:", (df["Unit_Price"] <= 0).sum())
print("Discount missing:", df["Discount"].isna().sum())
print("Discount < 0:", (df["Discount"] < 0).sum())
print("Discount = 0:", (df["Discount"] == 0).sum())
print("Discount > Unit_Price:", (df["Discount"] > df["Unit_Price"]).sum())

# A discount can exceed the single-unit price when the transaction contains
# multiple units, so this is audited but not treated as an error by itself.


Unit_Price missing: 0
Unit_Price <= 0: 0
Discount missing: 0
Discount < 0: 0
Discount = 0: 1
Discount > Unit_Price: 1120


### 3.7 Total Amount

In [21]:
calculated_total = (
    df["Quantity"] * df["Unit_Price"]
    - df["Discount"]
)

total_mismatch_mask = ~np.isclose(
    df["Total_Amount"],
    calculated_total,
    atol=0.01,
    rtol=0
)

print("Total_Amount <= 0:", (df["Total_Amount"] <= 0).sum())
print("Total_Amount < 0:", (df["Total_Amount"] < 0).sum())
print("Rows where Total_Amount does not match Quantity × Unit_Price - Discount:",
      total_mismatch_mask.sum())


Total_Amount <= 0: 1742
Total_Amount < 0: 1742
Rows where Total_Amount does not match Quantity × Unit_Price - Discount: 0


### 3.8 Transaction Date vs Encounter Date

In [22]:
if encounters_df is not None:
    encounter_dates = pd.to_datetime(
        encounters_df
        .drop_duplicates("Encounter_ID")
        .set_index("Encounter_ID")["Encounter_Date"],
        errors="coerce"
    )

    linked_encounter_date = df["Encounter_ID"].map(encounter_dates)

    print(
        "Transactions dated before linked encounter date:",
        (txn_date_audit < linked_encounter_date).sum()
    )

    # This is retained as an audit finding because a pharmacy transaction
    # may follow a prescription/encounter under business-specific workflows.
else:
    print("Encounter reference unavailable; date relationship audit skipped.")


Transactions dated before linked encounter date: 6934


## 4. Data Cleaning

### 4.1 Identifier Formatting

In [23]:
for col in ["Pharmacy_Txn_ID", "Patient_ID", "Medicine_ID", "Encounter_ID"]:
    df[col] = df[col].astype("string").str.strip()


### 4.2 Medicine_ID

In [24]:
medicine_id_cleaned = df["Medicine_ID"]

if medicines_df is not None:
    medicine_id_cleaned = medicine_id_cleaned.mask(
        ~medicine_id_cleaned.isin(valid_medicine_ids)
    )
else:
    medicine_id_cleaned = medicine_id_cleaned.mask(
        ~medicine_id_cleaned.str.fullmatch(r"M\d{4}", na=False)
    )

df["Medicine_ID"] = medicine_id_cleaned


### 4.3 Patient_ID

In [25]:
if encounters_df is not None:
    # Encounter_ID is the reliable relationship to the encounter master.
    # Use the encounter master Patient_ID to repair mismatched patient links.
    linked_patient = df["Encounter_ID"].map(encounter_patient_map)

    patient_cleaned = df["Patient_ID"].copy()
    patient_cleaned.loc[linked_patient.notna()] = linked_patient.loc[linked_patient.notna()]

    df["Patient_ID"] = patient_cleaned


### 4.4 Txn_Date

In [26]:
txn_date_cleaned = pd.to_datetime(
    df["Txn_Date"],
    errors="coerce"
)

df["Txn_Date"] = txn_date_cleaned


### 4.5 Quantity

In [28]:
quantity_cleaned = df["Quantity"].mask(df["Quantity"] <= 0)

df["Quantity"] = quantity_cleaned.astype("Int64")

### 4.6 Total_Amount

In [30]:
total_amount_cleaned = (
    df["Quantity"].astype("Float64")
    * df["Unit_Price"]
    - df["Discount"]
)

df["Total_Amount"] = total_amount_cleaned

## 5. Final Validation

In [31]:
print("Final shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nDuplicate Pharmacy_Txn_ID:", df["Pharmacy_Txn_ID"].duplicated().sum())
print("\nDtypes:")
print(df.dtypes)

Final shape: (14000, 9)

Missing values:
Pharmacy_Txn_ID       0
Patient_ID            0
Medicine_ID         139
Encounter_ID          0
Txn_Date              0
Quantity           1635
Unit_Price            0
Discount              0
Total_Amount       1635
dtype: int64

Duplicate rows: 0

Duplicate Pharmacy_Txn_ID: 0

Dtypes:
Pharmacy_Txn_ID    string[python]
Patient_ID         string[python]
Medicine_ID        string[python]
Encounter_ID       string[python]
Txn_Date           datetime64[ns]
Quantity                    Int64
Unit_Price                float64
Discount                  float64
Total_Amount              Float64
dtype: object


In [32]:
# Validate Medicine_ID
if medicines_df is not None:
    invalid_medicine_mask = (
        df["Medicine_ID"].notna()
        & ~df["Medicine_ID"].isin(valid_medicine_ids)
    )
else:
    invalid_medicine_mask = (
        df["Medicine_ID"].notna()
        & ~df["Medicine_ID"].astype("string").str.fullmatch(r"M\d{4}", na=False)
    )

print(
    "Invalid Medicine_ID after cleaning:",
    invalid_medicine_mask.sum()
)

# Validate Quantity
non_positive_quantity = (
    df["Quantity"].dropna() <= 0
).sum()

print(
    "Non-positive Quantity after cleaning:",
    non_positive_quantity
)

# Validate Total_Amount
negative_total_amount = (
    df["Total_Amount"].dropna() < 0
).sum()

print(
    "Negative Total_Amount after cleaning:",
    negative_total_amount
)

# Validate Total_Amount formula
# Only check rows where all required values are present
formula_rows = df[
    ["Quantity", "Unit_Price", "Discount", "Total_Amount"]
].notna().all(axis=1)

expected_total = (
    df.loc[formula_rows, "Quantity"].astype(float)
    * df.loc[formula_rows, "Unit_Price"].astype(float)
    - df.loc[formula_rows, "Discount"].astype(float)
)

formula_check = np.isclose(
    df.loc[formula_rows, "Total_Amount"].astype(float),
    expected_total.astype(float),
    atol=0.01,
    rtol=0
)

print(
    "Total_Amount formula mismatches after cleaning:",
    (~formula_check).sum()
)

Invalid Medicine_ID after cleaning: 0
Non-positive Quantity after cleaning: 0
Negative Total_Amount after cleaning: 107
Total_Amount formula mismatches after cleaning: 0


In [33]:
df.head()

,Pharmacy_Txn_ID,Patient_ID,Medicine_ID,Encounter_ID,Txn_Date,Quantity,Unit_Price,Discount,Total_Amount
0,PT500000,P103080,M1091,E205571,2025-10-11,13,1153.50,212.49,14783.01
1,PT500001,P106049,M1119,E208474,2026-04-01,<NA>,898.72,475.10,<NA>
2,PT500002,P102595,M1088,E213936,2026-08-16,27,2874.78,94.26,77524.8
3,PT500003,P103962,M1094,E212748,2025-08-29,24,1422.18,303.45,33828.87
4,PT500004,P100937,M1054,E201917,2025-07-07,3,1475.38,122.44,4303.7


## 6. Export Cleaned Data

In [35]:
df.to_csv(CLEANED_DIR / "08_pharmacy_transactions_cleaned.csv", index=False)